# 02 — Sales trends and revenue analysis dashboard (PowerBI)

Monthly revenue, order volume, average order value, seasonality.

**Show monthly sales trend, customer segmentation** The dataset shows growth in online sales. The dashboard will provide charts for sales growth across different regions, customer concentration and top products/categories.



In [ ]:
import os

import pandas as pd
import numpy as np

# from dotenv import load_dotenv
# from sqlalchemy import create_engine

# load_dotenv("../.env")

# # Marts only. The dialect keeps the warehouse swappable (§9).
# engine = create_engine(f"bigquery://{os.environ['GCP_PROJECT']}/olist_marts")
data_path=[]

for dirname, _, filenames in os.walk('/home/bwong/M2/ntu-dsai-group5-project2/data/staging/'):
    for filename in filenames:
       data_path.append(os.path.join(dirname, filename))


data_path

In [ ]:
base_path = "/home/bwong/M2/ntu-dsai-group5-project2/data/staging/"

orders = pd.read_csv(base_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(base_path + "olist_order_items_dataset.csv")
customers = pd.read_csv(base_path + "olist_customers_dataset.csv")
products = pd.read_csv(base_path + "olist_products_dataset.csv")
categories = pd.read_csv(base_path + "product_category_name_translation.csv")
order_payments = pd.read_csv(base_path + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(base_path + "olist_order_reviews_dataset.csv")



In [ ]:
order_items.head(10)

In [ ]:
customers.head(10)

In [ ]:
products.head(10)

In [ ]:
from google.cloud import bigquery

client = bigquery.Client()

# Define target BigQuery details
project_id = "dsai6mod2"
dataset_id = "dsdwh"
table_id = "sales_fact"

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

orders['org_to_carrier_time'] = orders['order_delivered_carrier_date'] - orders['order_approved_at']
orders['carrier_to_customer_time'] = orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']
orders['actual_arrival_time'] = orders['order_delivered_customer_date'] - orders['order_approved_at']
orders['apply_SLA'] = orders['order_delivered_customer_date'] <= orders['order_estimated_delivery_date']
orders['delivery_delay']= orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']

orders_sla = orders[orders['apply_SLA'] == True]

order_items=pd.merge(orders_sla,items,on='order_id')

#orders_customers = pd.merge(order_items,customers, on='customer_id',how='outer')
final_data = pd.merge(order_items,products,on='product_id')

final_data["order_purchase_timestamp"]=pd.to_datetime(final_data["order_purchase_timestamp"])
final_data['month']=final_data["order_purchase_timestamp"].dt.month
final_data['year']=final_data["order_purchase_timestamp"].dt.year

# Option 1: Using pandas-gbq
final_data.to_gbq(
    destination_table=f"{dataset_id}.{table_id}",
    project_id=project_id,
    if_exists="replace"  # options: 'fail', 'replace', 'append'
)
    
print("DataFrame loaded successfully into BigQuery!")


In [ ]:
order_items["order_purchase_timestamp"]=pd.to_datetime(order_items["order_purchase_timestamp"])
order_items['month']=order_items["order_purchase_timestamp"].dt.month
order_items['year']=order_items["order_purchase_timestamp"].dt.year

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

gmv=order_items.groupby('year')['price'].sum()
print(gmv)
plt.figure(figsize=(5,5))
plt.plot(gmv.index,gmv.values)
plt.title("GMV by Year")
plt.xlabel("Year")
plt.ylabel("GMV")
plt.grid(True)
plt.xticks(gmv.index)
plt.show()

In [ ]:
# from google.cloud import bigquery

# client = bigquery.Client()


# # Example DataFrame
# df = pd.DataFrame({
#     "customer_id": [1, 2, 3],
#     "customer_name": ["Alice", "Bob", "Charlie"],
#     "spend": [100.5, 200.75, 300.0]
# })

# # Define target BigQuery details
# project_id = "dsai6mod2"
# dataset_id = "dsdwh"
# table_id = "sales_trends"

# # Option 1: Using pandas-gbq
# df.to_gbq(
#     destination_table=f"{dataset_id}.{table_id}",
#     project_id=project_id,
#     if_exists="replace"  # options: 'fail', 'replace', 'append'
# )


# print("DataFrame loaded successfully into BigQuery!")
